# Smart Classroom Digital Twin: CO₂ Monitoring and Ventilation Optimization
**Selected option:** environmental system / ventilated room with CO₂. This educational simulation uses illustrative parameters, not Veltech sensor data.

In [ ]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED=42
def simulate(policy="smart", seed=SEED, generation=0.0048):
    rng=np.random.default_rng(seed); V=180.; outdoor=420.; c=420.; rows=[]
    for m in range(1,361):
        occupied=(20<=m<180) or (210<=m<330); people=28 if occupied else 0
        sensor=c+rng.normal(0,18)
        if policy=="smart":
            q=420 if sensor>=900 else 110
            if sensor>=1050: q=520
        elif policy=="low": q=110
        elif policy=="high": q=420
        else: raise ValueError("Use smart, low, or high")
        c=max(outdoor,c+(1/60)*(people*generation*1e6-q*(c-outdoor))/V)
        rows.append({"minute":m,"clock_hour":9+m/60,"occupied":int(occupied),"people":people,
                      "co2_ppm":c,"sensor_ppm":sensor,"ventilation_m3_h":q})
    return pd.DataFrame(rows)
smart, low, high = simulate("smart"), simulate("low"), simulate("high")
def metrics(d):
    o=d[d.occupied==1]
    return pd.Series({"Peak CO2 ppm":d.co2_ppm.max(),"Mean occupied CO2 ppm":o.co2_ppm.mean(),
                      "Occupied minutes above 1000 ppm":(o.co2_ppm>1000).sum(),
                      "Mean ventilation m3/h":d.ventilation_m3_h.mean()})
score=pd.DataFrame({"Smart control":metrics(smart),"Low fixed":metrics(low),"High fixed":metrics(high)}).T.round(1)
display(score)
plt.figure(figsize=(10,5))
for d,l in [(smart,"Smart control"),(low,"Low fixed"),(high,"High fixed")]: plt.plot(d.clock_hour,d.co2_ppm,label=l)
plt.axhline(1000,ls="--",label="Illustrative 1000 ppm reference");plt.xlabel("Clock time");plt.ylabel("CO2 ppm")
plt.title("CO2 comparison");plt.grid(alpha=.25);plt.legend();plt.show()
plt.figure(figsize=(10,4));plt.step(smart.clock_hour,smart.ventilation_m3_h,where="post");plt.xlabel("Clock time");plt.ylabel("Ventilation m3/h");plt.title("Controller decisions");plt.show()
plt.figure(figsize=(10,4));plt.plot(smart.clock_hour,smart.co2_ppm,label="Twin state");plt.plot(smart.clock_hour,smart.sensor_ppm,alpha=.5,label="Noisy sensor");plt.legend();plt.title("Sensor vs twin");plt.show()
# Calibration / identifiability demonstration using synthetic reference
ref=smart.co2_ppm.to_numpy(); rows=[]
for g in [0.0035,0.0042,0.0048,0.0055,0.0062]:
    candidate=simulate("smart",generation=g)
    rows.append({"generation coefficient":g,"RMSE ppm":np.sqrt(np.mean((candidate.co2_ppm.to_numpy()-ref)**2))})
display(pd.DataFrame(rows).sort_values("RMSE ppm"))
# Intelligence layer: sensor-jump flags for review, not automatic deletion
smart["sensor_jump_flag"]=smart.sensor_ppm.diff().abs()>120
print("Sensor jump flags:",int(smart.sensor_jump_flag.sum()))
# Verification
assert (smart.co2_ppm>=420).all()
assert np.allclose(smart.co2_ppm,simulate("smart").co2_ppm)
assert metrics(high)["Mean occupied CO2 ppm"] < metrics(low)["Mean occupied CO2 ppm"]
print("All verification tests passed.")
